# Crop Cnn Experiments

Ce notebook reprend le script `crop_cnn_experiments.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Modele crop image utilisable sur frame courante pour contexte attention/PPE.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Train CNN/transfer crop models for attention and blouse/PPE.
- Commande de reproduction referencee : crop CNN catalogue.
- Artefacts controles : Crop CNN catalogue exists. (`runs/exp_013_crop_cnn_catalogue/metrics/crop_cnn_metrics.csv`).
- Run par defaut : `runs/exp_013_crop_cnn_catalogue`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "crop_cnn_experiments.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
import math
import random
import time
from collections import Counter
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

from ml_pipeline import ROOT, RUNS_DIR, read_frame, safe_auc, write_json
from sequence_experiments import append_report, make_run_dir


TARGETS = {
    "attention": {"positive": "distracted", "negative": "attentive"},
    "blouse": {"positive": "badly_worn", "negative": "properly_worn"},
}


## Fonction `set_seed`

Cette cellule definit `set_seed`. Elle prepare une partie du script.

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## Fonction `crop_from_bbox`

Cette cellule definit `crop_from_bbox`. Elle prepare une partie du script.

In [ ]:
def crop_from_bbox(frame, row, pad=0.12):
    h, w = frame.shape[:2]
    x1 = float(row["bbox_x1"])
    y1 = float(row["bbox_y1"])
    x2 = float(row["bbox_x2"])
    y2 = float(row["bbox_y2"])
    bw = max(1.0, x2 - x1)
    bh = max(1.0, y2 - y1)
    x1 -= bw * pad
    x2 += bw * pad
    y1 -= bh * pad
    y2 += bh * pad
    x1 = int(max(0, min(w - 1, x1)))
    x2 = int(max(0, min(w, x2)))
    y1 = int(max(0, min(h - 1, y1)))
    y2 = int(max(0, min(h, y2)))
    if x2 <= x1 or y2 <= y1:
        return frame
    return frame[y1:y2, x1:x2]


## Fonction `extract_crop_dataset`

Cette cellule definit `extract_crop_dataset`. Elle prepare une partie du script.

In [ ]:
def extract_crop_dataset(base_run, run_dir, stride_frames, max_per_video):
    base = Path(base_run)
    if not base.is_absolute():
        base = ROOT / base
    pose = pd.read_csv(base / "features" / "pose_features.csv")
    split = pd.read_csv(base / "split.csv")
    split_by_video = {row["video_id"]: row for _, row in split.iterrows()}
    rows = []
    crop_dir = run_dir / "features" / "crops"
    crop_dir.mkdir(parents=True, exist_ok=True)
    for video_id, group in pose.groupby("video_id", sort=False):
        if video_id not in split_by_video:
            continue
        group = group.sort_values("frame")
        selected = group[group["frame"] % stride_frames == 0].copy()
        if len(selected) > max_per_video:
            selected = selected.iloc[np.linspace(0, len(selected) - 1, max_per_video).round().astype(int)]
        video_meta = split_by_video[video_id]
        for _, row in selected.iterrows():
            frame_idx = int(row["frame"])
            frame = read_frame(ROOT / row["path"], frame_idx)
            if frame is None:
                continue
            crop = crop_from_bbox(frame, row)
            out_path = crop_dir / f"{video_id}_{frame_idx:06d}.jpg"
            cv2.imwrite(str(out_path), crop)
            rows.append(
                {
                    "video_id": video_id,
                    "path": str(out_path.relative_to(run_dir)),
                    "source_video_path": row["path"],
                    "frame": frame_idx,
                    "time_s": float(row["time_s"]),
                    "split": video_meta["split"],
                    "attention": video_meta["attention"],
                    "attention_label": int(video_meta["attention"] == TARGETS["attention"]["positive"]),
                    "blouse": video_meta["blouse"],
                    "blouse_label": int(video_meta["blouse"] == TARGETS["blouse"]["positive"]),
                }
            )
    df = pd.DataFrame(rows)
    df.to_csv(run_dir / "features" / "crop_cnn_index.csv", index=False)
    audit = {
        "base_run": str(base),
        "samples": int(len(df)),
        "videos": int(df["video_id"].nunique()) if len(df) else 0,
        "stride_frames": int(stride_frames),
        "max_per_video": int(max_per_video),
        "split_counts": dict(Counter(df["split"])) if len(df) else {},
        "attention_counts": dict(Counter(df["attention"])) if len(df) else {},
        "blouse_counts": dict(Counter(df["blouse"])) if len(df) else {},
    }
    write_json(run_dir / "metrics" / "crop_dataset_audit.json", audit)
    append_report(
        run_dir,
        "Crop Dataset Build",
        f"- Samples: `{audit['samples']}`\n- Videos: `{audit['videos']}`\n- Split counts: `{audit['split_counts']}`",
    )
    return df


## Classe `CropDataset`

Cette cellule definit `CropDataset`. Elle prepare une partie du script.

In [ ]:
class CropDataset(Dataset):
    def __init__(self, run_dir, index, target, train=False, image_size=224):
        self.run_dir = Path(run_dir)
        self.index = index.reset_index(drop=True)
        self.target = target
        self.train = train
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        if train:
            self.tf = transforms.Compose(
                [
                    transforms.RandomResizedCrop(image_size, scale=(0.80, 1.0), ratio=(0.80, 1.25)),
                    transforms.RandomHorizontalFlip(p=0.25),
                    transforms.RandomApply([transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.18, hue=0.03)], p=0.75),
                    transforms.RandomRotation(degrees=6),
                    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.20),
                    transforms.ToTensor(),
                    normalize,
                ]
            )
        else:
            self.tf = transforms.Compose([transforms.Resize((image_size, image_size)), transforms.ToTensor(), normalize])

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        row = self.index.iloc[idx]
        path = self.run_dir / row["path"]
        image = Image.open(path).convert("RGB")
        x = self.tf(image)
        y = torch.tensor(float(row[f"{self.target}_label"]), dtype=torch.float32)
        return x, y


## Classe `SmallCropCNN`

Cette cellule definit `SmallCropCNN`. Elle prepare une partie du script.

In [ ]:
class SmallCropCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))

    def forward(self, x):
        return self.head(self.features(x)).squeeze(1)


## Fonction `make_model`

Cette cellule definit `make_model`. Elle prepare une partie du script.

In [ ]:
def make_model(arch, pretrained=True):
    if arch == "small_cnn":
        return SmallCropCNN()
    if arch == "mobilenet_v3_small":
        weights = None
        if pretrained:
            try:
                weights = models.MobileNet_V3_Small_Weights.DEFAULT
            except Exception:
                weights = None
        model = models.mobilenet_v3_small(weights=weights)
        for param in model.features.parameters():
            param.requires_grad = False
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, 1)
        return model
    if arch == "resnet18":
        weights = None
        if pretrained:
            try:
                weights = models.ResNet18_Weights.DEFAULT
            except Exception:
                weights = None
        model = models.resnet18(weights=weights)
        for name, param in model.named_parameters():
            if not name.startswith("layer4") and not name.startswith("fc"):
                param.requires_grad = False
        model.fc = nn.Linear(model.fc.in_features, 1)
        return model
    raise ValueError(arch)


## Fonction `predict`

Cette cellule definit `predict`. Elle prepare une partie du script.

In [ ]:
@torch.no_grad()
def predict(model, loader, device):
    model.eval()
    probs = []
    ys = []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb).view(-1)
        probs.append(torch.sigmoid(logits).detach().cpu().numpy())
        ys.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(ys)


## Fonction `train_crop_model`

Cette cellule definit `train_crop_model`. Elle prepare une partie du script.

In [ ]:
def train_crop_model(run_dir, index, target, arch, args, device):
    train_df = index[index["split"] == "train"].copy()
    val_df = index[index["split"] == "val"].copy()
    train_ds = CropDataset(run_dir, train_df, target, train=True, image_size=args.image_size)
    val_ds = CropDataset(run_dir, val_df, target, train=False, image_size=args.image_size)
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, num_workers=0)
    model = make_model(arch, pretrained=not args.no_pretrained).to(device)
    y_train = train_df[f"{target}_label"].to_numpy()
    positives = max(1, int(y_train.sum()))
    negatives = max(1, int(len(y_train) - positives))
    pos_weight = torch.tensor([negatives / positives], dtype=torch.float32, device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=args.lr, weight_decay=args.weight_decay)
    best = {"ap": -1.0, "state": None, "epoch": 0}
    history = []
    patience_left = args.patience
    start = time.perf_counter()
    for epoch in range(1, args.epochs + 1):
        model.train()
        losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb).view(-1), yb)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        val_probs, val_y = predict(model, val_loader, device)
        val_ap = safe_auc(average_precision_score, val_y.astype(int), val_probs)
        val_score = float(val_ap or 0.0)
        history.append({"target": target, "architecture": arch, "epoch": epoch, "train_loss": float(np.mean(losses)), "val_ap": val_score})
        if val_score > best["ap"] + 1e-5:
            best = {"ap": val_score, "state": {k: v.detach().cpu() for k, v in model.state_dict().items()}, "epoch": epoch}
            patience_left = args.patience
        else:
            patience_left -= 1
        if patience_left <= 0:
            break
    if best["state"] is not None:
        model.load_state_dict(best["state"])
    train_time = time.perf_counter() - start
    model_path = run_dir / "models" / f"{target}_{arch}.pt"
    torch.save({"target": target, "architecture": arch, "state_dict": model.state_dict(), "best_epoch": best["epoch"]}, model_path)
    return model, history, train_time, model_path.stat().st_size


## Fonction `video_metrics`

Cette cellule definit `video_metrics`. Elle prepare une partie du script.

In [ ]:
def video_metrics(pred_df, target):
    rows = []
    for split, group in pred_df.groupby("split"):
        video = group.groupby("video_id", as_index=False).agg(label=(f"{target}_label", "max"), risk=("risk", "mean"))
        y = video["label"].astype(int).to_numpy()
        p = video["risk"].to_numpy()
        best = None
        for threshold in np.arange(0.05, 1.0, 0.05):
            pred = (p >= threshold).astype(int)
            score = f1_score(y, pred, zero_division=0)
            candidate = {
                "threshold": float(threshold),
                "f1": float(score),
                "accuracy": float(accuracy_score(y, pred)),
                "balanced_accuracy": float(balanced_accuracy_score(y, pred)) if len(np.unique(y)) > 1 else None,
                "confusion_matrix": confusion_matrix(y, pred, labels=[0, 1]).tolist(),
            }
            if best is None or candidate["f1"] > best["f1"]:
                best = candidate
        rows.append(
            {
                "split": split,
                "n_videos": int(len(video)),
                "positive_videos": int(y.sum()),
                "average_precision": safe_auc(average_precision_score, y, p),
                "roc_auc": safe_auc(roc_auc_score, y, p),
                **best,
            }
        )
    return rows


## Fonction `make_contact_sheet`

Cette cellule definit `make_contact_sheet`. Elle prepare une partie du script.

In [ ]:
def make_contact_sheet(run_dir, index):
    out_dir = run_dir / "error_review" / "crop_cnn_examples"
    out_dir.mkdir(parents=True, exist_ok=True)
    sample = index[index["split"] == "train"].head(12)
    tiles = []
    for _, row in sample.iterrows():
        img = cv2.imread(str(run_dir / row["path"]))
        if img is None:
            continue
        img = cv2.resize(img, (160, 160))
        cv2.putText(img, row["attention"], (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(img, row["blouse"], (8, 145), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 2, cv2.LINE_AA)
        tiles.append(img)
    if tiles:
        rows = []
        for i in range(0, len(tiles), 4):
            chunk = tiles[i : i + 4]
            while len(chunk) < 4:
                chunk.append(np.full_like(tiles[0], 255))
            rows.append(np.hstack(chunk))
        cv2.imwrite(str(out_dir / "crop_training_examples.jpg"), np.vstack(rows))


## Fonction `run_catalogue`

Cette cellule definit `run_catalogue`. Elle prepare une partie du script.

In [ ]:
def run_catalogue(args):
    set_seed(args.seed)
    run_dir = make_run_dir(args.run_name)
    write_json(
        run_dir / "config.json",
        {
            "base_run": args.base_run,
            "seed": args.seed,
            "architectures": args.architectures,
            "epochs": args.epochs,
            "image_size": args.image_size,
            "pretrained": not args.no_pretrained,
            "created_at": datetime.now().isoformat(timespec="seconds"),
        },
    )
    index = extract_crop_dataset(args.base_run, run_dir, args.stride_frames, args.max_per_video)
    make_contact_sheet(run_dir, index)
    device = torch.device("cuda" if torch.cuda.is_available() and args.device == "auto" else args.device)
    all_history = []
    all_metrics = []
    all_predictions = []
    for target in TARGETS:
        for arch in args.architectures:
            print(f"training {target} {arch} on {device}")
            model, history, train_time_s, model_size = train_crop_model(run_dir, index, target, arch, args, device)
            all_history.extend(history)
            eval_loader = DataLoader(CropDataset(run_dir, index, target, train=False, image_size=args.image_size), batch_size=args.batch_size, shuffle=False, num_workers=0)
            probs, _ = predict(model, eval_loader, device)
            pred_df = index[["video_id", "split", "frame", "time_s", f"{target}_label"]].copy()
            pred_df["target"] = target
            pred_df["architecture"] = arch
            pred_df["risk"] = probs
            pred_df.to_csv(run_dir / "features" / f"predictions_{target}_{arch}.csv", index=False)
            all_predictions.append(pred_df)
            for row in video_metrics(pred_df, target):
                row.update({"target": target, "architecture": arch, "train_time_s": train_time_s, "model_size_bytes": model_size})
                all_metrics.append(row)
            pd.DataFrame(all_metrics).to_csv(run_dir / "metrics" / "crop_cnn_metrics.csv", index=False)
            pd.DataFrame(all_history).to_csv(run_dir / "metrics" / "crop_cnn_training_history.csv", index=False)
    if all_predictions:
        pd.concat(all_predictions, ignore_index=True).to_csv(run_dir / "features" / "crop_cnn_all_predictions.csv", index=False)
    metrics = pd.DataFrame(all_metrics)
    metrics.to_csv(run_dir / "metrics" / "crop_cnn_metrics.csv", index=False)
    lines = ["# Crop CNN Catalogue", ""]
    lines.append("| target | architecture | split | AP | ROC AUC | best F1 | threshold | balanced accuracy |")
    lines.append("|---|---|---|---:|---:|---:|---:|---:|")
    for _, row in metrics.sort_values(["target", "split", "average_precision"], ascending=[True, True, False]).iterrows():
        lines.append(
            f"| {row['target']} | {row['architecture']} | {row['split']} | {row['average_precision'] if pd.notna(row['average_precision']) else 'NA'} | {row['roc_auc'] if pd.notna(row['roc_auc']) else 'NA'} | {row['f1']:.3f} | {row['threshold']:.2f} | {row['balanced_accuracy'] if pd.notna(row['balanced_accuracy']) else 'NA'} |"
        )
    (run_dir / "crop_cnn_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    append_report(run_dir, "Crop CNN Catalogue Completion", f"- Metrics: `{run_dir / 'metrics' / 'crop_cnn_metrics.csv'}`\n- Summary: `{run_dir / 'crop_cnn_summary.md'}`")
    print(run_dir)


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Train CNN/transfer crop models for attention and blouse/PPE.")
    parser.add_argument("--base-run", default="runs/exp_006_final_research")
    parser.add_argument("--run-name", default="exp_013_crop_cnn_catalogue")
    parser.add_argument("--architectures", nargs="+", default=["small_cnn", "mobilenet_v3_small", "resnet18"])
    parser.add_argument("--stride-frames", type=int, default=15)
    parser.add_argument("--max-per-video", type=int, default=24)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--epochs", type=int, default=18)
    parser.add_argument("--patience", type=int, default=5)
    parser.add_argument("--batch-size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=3e-4)
    parser.add_argument("--weight-decay", type=float, default=1e-4)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto")
    parser.add_argument("--no-pretrained", action="store_true")
    args = parser.parse_args()
    run_catalogue(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_013_crop_cnn_catalogue_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["crop_cnn_experiments.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
